# compare

> Fan out one PDF across N engines and collect a `ComparisonResultList`.
> Two modes: spawn local pinned instances (Mode A), or point at a
> pre-existing `{engine: url}` map of already-running instances (Mode B).

This is where everything from the first three notebooks comes together.
`compare(pdf_path, page_range, engines=[...])` is the one function most
callers of this package will actually use: give it a PDF and a list of
engine names, and it hands back a `ComparisonResultList` (`02_report`) with
one row per engine, built by talking to each engine's `estravon-backend`
instance through a `Client` (`01_client`).

We've already built every individual piece: `get_artusi()` gives us a real
PDF to test against, `Client` and `LocalEngineProcess` know how to talk to
one running backend, and `ComparisonResultList` knows how to display the
results afterward. `compare()` is the loop that ties them together -- for
each engine we want, start or find a server, submit the PDF, collect the
result, and move on to the next engine, even if this one failed.

In [1]:
#| default_exp compare

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
from __future__ import annotations

from estravon_bench.client import Client, LocalEngineProcess
from estravon_bench.io import ComparisonResult
from estravon_bench.report import ComparisonResultList

_DEFAULT_BASE_PORT = 7860

## One engine at a time: `_run_one()`

Before looping over several engines, we need the code for handling just
one. `_run_one()` takes a `Client` that is already pointed at a reachable
server (it doesn't care whether that server came from `LocalEngineProcess`
or was already running -- that decision happens one level up, in
`compare()`) and turns whatever happens into a `ComparisonResult`:

- If the submission comes back `"done"`, we fetch the actual Markdown text
  (the two-step fetch from `01_client`) and build a successful result --
  engine name, text, timing, cost, page count.
- If it comes back anything else -- an error status, or an exception raised
  anywhere along the way (a network failure, a timeout, a malformed
  response) -- we catch it and build a failed result instead, with `error`
  set to whatever went wrong.

Either way, `_run_one()` itself never raises. That's the property `compare()`
depends on to keep going after one engine fails instead of stopping the
whole comparison.

In [4]:
#| export
def _run_one(
    client: Client, engine: str, pdf_path: str, section_name: str, page_range: str,
    chunk_size: int, mode: str, force_ocr: bool, max_wait_s: float,
) -> ComparisonResult:
    """Submit to one already-reachable Client and turn the result into a
    ComparisonResult -- never raises, a failure becomes an error row."""
    try:
        result = client.submit_and_wait(
            pdf_path, section_name, page_range,
            chunk_size=chunk_size, mode=mode, force_ocr=force_ocr, max_wait_s=max_wait_s,
        )
        if result.get("status") != "done":
            return ComparisonResult(engine=engine, backend_url=client.base_url,
                                     error=result.get("error") or f"status={result.get('status')}")
        parts = client.fetch_markdown(result)
        markdown = "\n\n".join(p["markdown"] for p in parts)
        return ComparisonResult(
            engine=engine, markdown=markdown,
            predict_time_s=result.get("total_predict_time_s"),
            cost_usd=result.get("total_cost_usd"),
            local=bool(result.get("local", False)),
            page_count=result.get("page_count"),
            backend_url=client.base_url,
        )
    except Exception as exc:
        return ComparisonResult(engine=engine, backend_url=client.base_url, error=str(exc))

## Several engines at once: `compare()`

`compare()` runs the loop, but before it can loop it needs to know how to
reach each engine's server. There are two ways we might already have that,
and `compare()` asks us to pick exactly one, not a mix of both:

- **We give it a list of engine names** (`engines=["mineru", "mistral"]`) and
  let it start a server for each one itself, using `LocalEngineProcess` from
  `01_client` -- this is "Mode A." It's the common case: we don't have
  anything running yet, we just want a comparison.
- **We already have servers running somewhere** -- maybe started earlier,
  maybe on another machine -- and give `compare()` their addresses directly
  (`engine_urls={"mistral": "http://host:7767"}`) -- this is "Mode B." No
  `LocalEngineProcess` is involved; `compare()` just talks to whatever we
  point it at.

Passing both, or neither, is a mistake we want to catch immediately rather
than have it silently do the wrong thing, so `compare()` checks this first
and raises `ValueError` right away if we get it wrong.

From there the two modes both do the same thing, just sourced differently:
for each engine, get a `Client` pointed at a reachable server (spawning one
in Mode A, reusing the given URL in Mode B), call `_run_one()`, and append
whatever it returns -- a success or a failure, it doesn't matter, both are
valid rows -- to the results we'll return. In Mode A, if the server itself
never becomes reachable (the subprocess fails to start, or never answers
`/ping` in time), that also becomes a failed row for that engine rather than
stopping the other engines from being tried.

In [6]:
#| export
def compare(
    pdf_path: str,
    page_range: str,
    engines: list[str] | None = None,
    engine_urls: dict[str, str] | None = None,
    section_name: str = "bench",
    chunk_size: int = 80,
    mode: str = "balanced",
    force_ocr: bool = False,
    api_key: str | None = None,
    max_wait_s: float = 300.0,
    ready_timeout_s: float = 60.0,
    base_port: int = _DEFAULT_BASE_PORT,
    estravon_cmd: str = "estravon",
) -> ComparisonResultList:
    """Run one PDF through multiple engines and return a side-by-side ComparisonResultList.

    Exactly one of `engines` (Mode A: spawn one pinned local `estravon`
    subprocess per engine -- no estravon-backend core change needed, drives
    the existing `--backend`/`--port` CLI flags) or `engine_urls` (Mode B: an
    explicit `{engine: base_url}` map of already-running instances) must be
    given.

    One engine failing, being unconfigured, or its subprocess failing to
    start yields an error row for that engine -- it never aborts the whole
    comparison. Local engines report `cost_usd=0.0, local=True` -- see
    ComparisonResultList.to_markdown_table() for how that's labelled.
    """
    if (engines is None) == (engine_urls is None):
        raise ValueError("pass exactly one of engines= (Mode A) or engine_urls= (Mode B)")

    results = ComparisonResultList()

    if engine_urls is not None:
        for engine, url in engine_urls.items():
            client = Client(url, api_key=api_key)
            results.append(_run_one(client, engine, pdf_path, section_name, page_range,
                                     chunk_size, mode, force_ocr, max_wait_s))
        return results

    for i, engine in enumerate(engines):
        port = base_port + i
        try:
            with LocalEngineProcess(engine, port, ready_timeout_s=ready_timeout_s,
                                     estravon_cmd=estravon_cmd) as proc:
                client = Client(proc.base_url, api_key=None)
                results.append(_run_one(client, engine, pdf_path, section_name, page_range,
                                         chunk_size, mode, force_ocr, max_wait_s))
        except Exception as exc:
            results.append(ComparisonResult(engine=engine, error=f"could not start instance: {exc}"))
    return results

### Try it

Two things worth seeing work before trusting `compare()` on a real
comparison. First, the argument checking: we call it with neither
`engines=` nor `engine_urls=`, then with both -- both should be rejected
with a clear `ValueError` rather than doing something unpredictable with
whichever argument happened to be `None`.

Second, the error-row behaviour, which is the whole point of `_run_one()`
existing separately from `compare()`: we point a `Client` at a fake server
that always answers with a 500 error, run `_run_one()` against it directly
(this is exactly what one iteration of `compare()`'s own loop does
internally), and check that we get back a normal `ComparisonResult` with
`error` set, not an exception thrown at us. This is what lets `compare()`
keep going and try the next engine when one of them is unreachable, rather
than the whole comparison stopping because of a single bad engine.

In [7]:
#| hide
# Mode-argument validation, and Mode B's error-row-not-abort behaviour --
# exercised against a fake transport, no live server or `estravon` binary needed.
import httpx as _httpx
import tempfile as _tf

try:
    compare("x.pdf", "1-1")
    raise AssertionError("should have raised: neither engines nor engine_urls given")
except ValueError as exc:
    print(f"compare() with neither engines= nor engine_urls= correctly raised: {exc}")

try:
    compare("x.pdf", "1-1", engines=["mineru"], engine_urls={"mineru": "http://x"})
    raise AssertionError("should have raised: both engines and engine_urls given")
except ValueError as exc:
    print(f"compare() with both engines= and engine_urls= correctly raised: {exc}")


def _fake_transport(request: _httpx.Request) -> _httpx.Response:
    if "/process" in request.url.path:
        return _httpx.Response(500, json={"status": "error", "error": "engine unavailable"})
    raise AssertionError(f"unexpected path {request.url.path}")


# compare() itself constructs its own Client with no transport override (real
# network, by design -- it's a thin loop over _run_one()), so the error-row
# behaviour is exercised at the _run_one() level directly, against an injected
# fake transport. This is what compare()'s Mode B loop body does internally.
with _tf.NamedTemporaryFile(suffix=".pdf") as f:
    f.write(b"%PDF-1.4 stub")
    f.flush()
    fake_client = Client("http://fake", transport=_httpx.MockTransport(_fake_transport))
    row = _run_one(fake_client, "broken_engine", f.name, "bench", "1-1", 80, "balanced", False, 30.0)
    fake_client.close()

assert not row.ok
assert row.engine == "broken_engine"
assert "engine unavailable" in row.error
print(row)

compare() with neither engines= nor engine_urls= correctly raised: pass exactly one of engines= (Mode A) or engine_urls= (Mode B)
compare() with both engines= and engine_urls= correctly raised: pass exactly one of engines= (Mode A) or engine_urls= (Mode B)
ComparisonResult(engine='broken_engine', markdown=None, predict_time_s=None, cost_usd=None, local=False, page_count=None, backend_url='http://fake', error='submit failed (500): engine unavailable')


---
That's the whole package: `get_artusi()` gives us a PDF, `compare()` runs it
through however many engines we ask for, and the `ComparisonResultList` it
returns knows how to show us the results. See
[`index`](index.ipynb) for a worked example against a real backend.

In [8]:
#| hide
import nbdev; nbdev.nbdev_export()